Para cada estación del metro en la Caracas medimos cuánta gente de TransMilenio hay hoy en su zona en hora pico.

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
validaciones=pd.read_csv("../data/processed/validaciones_calle53.csv",dtype={"codigo":str})
validaciones["fecha"]=pd.to_datetime(validaciones["fecha"])


Juntamos la estación Calle 13 (09110) con la temporal Av. Jiménez (09128) porque están en el mismo cruce. Dejamos noviembre (la Calle 22 estuvo cerrada en octubre) y las horas pico, y sacamos el promedio diario por estación.

In [3]:
validaciones.loc[validaciones["codigo"]=="09110","codigo"]="09128"
noviembre=validaciones[validaciones["fecha"]>="2026-11-01"]
pico=noviembre[noviembre["hora"].isin([6,7,8,17,18,19])]
pico_dia=pico.groupby(["codigo","fecha"])["validaciones"].sum().reset_index()
demanda=pico_dia.groupby("codigo")["validaciones"].mean().round().reset_index(name="validaciones_pico")

In [4]:
noviembre["fecha"].unique()

<DatetimeArray>
['2026-11-03 00:00:00', '2026-11-04 00:00:00', '2026-11-05 00:00:00',
 '2026-11-06 00:00:00', '2026-11-09 00:00:00', '2026-11-10 00:00:00',
 '2026-11-11 00:00:00', '2026-11-12 00:00:00', '2026-11-13 00:00:00']
Length: 9, dtype: datetime64[us]

Estaciones que hoy atienden la zona de cada estación del metro: la que está en el punto o, si no hay, las dos más cercanas. La conexión sale de los planos oficiales.

In [5]:
zonas=pd.DataFrame([
    ["Estación 11","Calle 2 - Calle 3","directa","Avenida Jimenez","09128"],
    ["Estación 11","Calle 2 - Calle 3","directa","Hortua","09107"],
    ["Estación 12","Calle 11 - Calle 13","directa","Avenida Jimenez","09128"],
    ["Estación 13","Calle 24A - Calle 26","cercana","Calle 26","09114"],
    ["Estación 14","Calle 42 - Calle 44","directa","Av. 39","09130"],
    ["Estación 14","Calle 42 - Calle 44","directa","Calle 53","09131"],
    ["Estación 15","Calle 61 - Calle 63","directa","Calle 57","09125"],
    ["Estación 15","Calle 61 - Calle 63","directa","Flores","09127"],
    ["Estación 16","Calle 72 - Calle 74","directa","Calle 72","09122"],
],columns=["estacion_metro","direccion","integracion","estacion_referencia","codigo"])

In [6]:
detalle=zonas.merge(demanda,on="codigo",how="left",validate="m:1")
detalle["validaciones_pico"].isna().sum()

np.int64(0)

Promedio por estación del metro (donde hay dos referencias suponemos que la gente se reparte igual).

In [7]:
nodos=detalle.groupby(["estacion_metro","direccion","integracion"])["validaciones_pico"].mean().round().reset_index()
nodos=nodos.sort_values("validaciones_pico",ascending=False)
nodos

,estacion_metro,direccion,integracion,validaciones_pico
5,Estación 16,Calle 72 - Calle 74,directa,12975.0
3,Estación 14,Calle 42 - Calle 44,directa,8536.0
1,Estación 12,Calle 11 - Calle 13,directa,7457.0
4,Estación 15,Calle 61 - Calle 63,directa,7200.0
2,Estación 13,Calle 24A - Calle 26,cercana,6999.0
0,Estación 11,Calle 2 - Calle 3,directa,5443.0


In [8]:
aportes=detalle.merge(detalle.groupby("estacion_metro").size().reset_index(name="n"),on="estacion_metro",validate="m:1")
aportes["aporte"]=(aportes["validaciones_pico"]/aportes["n"]).round()
arbol=px.treemap(aportes,path=["estacion_metro","estacion_referencia"],values="aporte",color="integracion",title="Demanda en hora pico alrededor de cada estación del metro y de dónde sale")
arbol.update_traces(textinfo="label+value")

La estación 16 es el nodo crítico: casi 13.000 validaciones en hora pico, un 50 % más que la siguiente, y conexión directa con el metro. Las estaciones 14, 12 y 15 tienen demandas parecidas. La 13 también, pero su conexión es cercana. La 11 es la de menos demanda. En la 11, 14 y 15 usamos el promedio de las dos estaciones más cercanas.

Conclusión: en las reaperturas la gente se pasó a la estación más cercana. Por eso esperamos que con el metro la demanda se concentre en las estaciones con conexión directa, empezando por la zona de la Calle 72.

In [9]:
nodos.to_csv("../data/processed/ranking_metro.csv",index=False)
detalle.to_csv("../data/processed/zonas_metro.csv",index=False)
demanda.to_csv("../data/processed/demanda_pico.csv",index=False)